# 06 Streamlit App
---

## Información

### Objetivo

Implementar y validar la interfaz conversacional de ALESSIA utilizando Streamlit, integrando el pipeline RAG desarrollado en las etapas anteriores para ofrecer una experiencia de consulta clara, intuitiva y orientada al ciudadano.

### Objetivos específicos

- Configurar el entorno necesario para ejecutar la aplicación Streamlit.
- Implementar la capa de presentación de ALESSIA utilizando Streamlit.
- Integrar la interfaz con el servicio que orquesta el pipeline RAG.
- Permitir que el usuario realice consultas mediante una interfaz conversacional.
- Mostrar las respuestas generadas por ALESSIA.
- Validar el funcionamiento de la aplicación como punto de acceso al sistema RAG.

### Entradas

- Consulta realizada por el usuario.
- Servicio de aplicación (`AgentService`).
- Pipeline RAG implementado en las etapas anteriores.
- Base vectorial persistida en ChromaDB.
- Modelos de Embeddings y Reranking.
- Configuración del proveedor LLM.

### Salidas

- Interfaz conversacional funcional desarrollada con Streamlit.
- Respuestas generadas por ALESSIA a partir de consultas del usuario.
- Validación de la integración entre la interfaz de usuario y el pipeline RAG.

---

## Etapa 1. Preparación de la aplicación

En esta etapa se configura el entorno de ejecución de la aplicación y se preparan los componentes necesarios para conectar la interfaz Streamlit con el pipeline RAG de ALESSIA.

### 1. Configuración del entorno

Se configura el entorno de trabajo para permitir la importación de los módulos internos del proyecto.

Se agrega la raíz del proyecto al `PATH` para acceder a los componentes ubicados dentro de la carpeta `src`.

In [28]:
from pathlib import Path
import sys

# Obtener la raíz del proyecto
project_root = Path.cwd().parent

# Agregar la raíz del proyecto al PATH
sys.path.append(str(project_root))

# Ignorar warnings
import warnings
warnings.filterwarnings("ignore")

### 2. Importación de componentes

Se importan los módulos necesarios para inicializar la aplicación y conectar la interfaz con el pipeline RAG.

En esta etapa se prepara la comunicación entre la capa de presentación y los servicios internos del sistema.

In [29]:
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

from src.processing.vector_store import (
    get_or_create_collection,
)

from src.services.agent_service import (
    AgentService,
)

### 3. Carga del Vector Store y modelos

Se inicializan los componentes utilizados por el pipeline RAG.

Estos recursos se cargarán una sola vez durante la ejecución de la aplicación y posteriormente serán reutilizados por `AgentService`.

In [30]:
vector_store_path = (
    project_root 
    / "data"
    / "vector_store"
)

collection = get_or_create_collection(
    collection_name="alessia_collection",
    vector_store_path=vector_store_path,
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reranker_model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4227.97it/s]


### 4. Inicialización de AgentService

Se crea la instancia del servicio de aplicación que actuará como punto de acceso al pipeline RAG.

La interfaz Streamlit no interactuará directamente con los módulos de Retrieval, Reranking o Generation, sino que utilizará este servicio como capa de abstracción.

In [31]:
agent_service = AgentService(
    collection=collection,
    embedding_model=embedding_model,
    reranker_model=reranker_model,
)

---
## Etapa 2. Integración del pipeline RAG
En esta etapa se integra AgentService con los componentes internos del pipeline RAG.

El objetivo es validar que una consulta realizada por el usuario puede recorrer todas las capas del sistema.

### 1. Definir consulta de prueba

Se define una consulta que será utilizada para validar la comunicación entre la capa de aplicación y el pipeline RAG.

La consulta representa una interacción real que podría realizar un ciudadano utilizando ALESSIA.

In [32]:
question = """
¿Qué medidas debe tomar la comunidad ante una emergencia por inundación?
"""

### 2. Ejecutar consulta mediante AgentService

Se envía la consulta al servicio de aplicación.

AgentService actúa como punto único de acceso al pipeline RAG, ocultando los detalles internos de recuperación, reranking y generación.

In [33]:
answer = agent_service.ask(
    question
)

print(answer)

None


---
## Etapa 3. Implementación de AgentService

### 1. Implementar Ask()
Este será el orquestador

In [34]:
def ask(
    self,
    question: str,
) -> str:
    """
    Procesa una consulta utilizando el pipeline RAG.

    Parameters
    ----------
    question : str
        Consulta realizada por el usuario.

    Returns
    -------
    str
        Respuesta generada por el modelo de lenguaje.
    """

    chunks = self._retrieve_context(
        question
    )

    reranked_chunks = self._rerank_chunks(
        question,
        chunks,
    )

    answer = self._generate_answer(
        question,
        reranked_chunks,
    )

    return answer

## 2. Implementar _retrieve_context()
Ahora conectamos con el módulo existente

In [35]:
from src.processing.retriever import retrieve_context
from src.models.chunk import Chunk

In [36]:
def _retrieve_context(
    self,
    question: str,
) -> list[Chunk]:
    """
    Recupera los fragmentos más relevantes
    para una consulta.
    """

    return retrieve_context(
        query=question,
        collection=self.collection,
        model=self.embedding_model,
        k=5,
    )

### 3. Implementar _rerank_chunks()

In [37]:
from src.processing.reranker import rerank_chunks

In [38]:
def _rerank_chunks(
    self,
    question: str,
    chunks: list[Chunk],
) -> list[Chunk]:
    """
    Reordena los fragmentos recuperados
    utilizando el modelo CrossEncoder.
    """

    return rerank_chunks(
        query=question,
        chunks=chunks,
        model=self.reranker_model,
        top_k=3,
    )

### 4. Implementar _generate_answer()

In [39]:
from src.processing.generation import generate_answer

In [40]:
def _generate_answer(
    self,
    question: str,
    context: list[Chunk],
) -> str:
    """
    Genera una respuesta utilizando
    el contexto recuperado.
    """

    return generate_answer(
        question=question,
        context=context,
    )

---
## Etapa 3. Construcción de la interface conversacional

Implementar la interfaz conversacional de ALESSIA utilizando Streamlit, permitiendo que el usuario realice consultas y visualice las respuestas generadas por el sistema.

La interfaz actuará como capa de presentación, delegando la lógica del procesamiento al servicio de aplicación.

### 1. Configuración de la aplicación

Se configura la aplicación Streamlit definiendo los parámetros generales de la interfaz.

En esta etapa se establece el título, icono y comportamiento inicial de la aplicación.

In [41]:
import streamlit as st

st.set_page_config(
    page_title="ALESSIA",
    page_icon="🌊",
    layout="centered",
)

2026-07-19 03:28:29.296 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### 2. Identidad visual

Se define la identidad inicial de ALESSIA dentro de la interfaz.

Posteriormente esta sección incorporará elementos gráficos como logo e imágenes del asistente.

In [46]:
st.title(
    "ALESSIA"
)

st.write(
    "Asistente inteligente para la gestión del riesgo de desastres en Valdivia."
)

2026-07-19 03:28:48.014 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:48.019 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:48.020 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:48.021 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:48.022 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:48.023 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### 3. Entrada del usuario

Se crea el componente de entrada que permitirá al usuario realizar consultas al asistente.

In [47]:
question = st.chat_input(
    "Escribe tu consulta..."
)

2026-07-19 03:28:58.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:58.047 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:58.060 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:58.061 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-19 03:28:58.061 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### 4. Integración con AgentService

Se importa el servicio de aplicación encargado de comunicarse con el pipeline RAG.

La interfaz Streamlit no ejecuta directamente las etapas de Retrieval, Reranking o Generation, sino que utiliza AgentService como punto único de acceso al sistema.

In [48]:
from src.services.agent_service import AgentService

### 5. Carga de componentes del sistema

Se inicializan los recursos utilizados por AgentService.

Los componentes se almacenan utilizando mecanismos de caché de Streamlit para evitar cargas repetidas durante la interacción del usuario.

In [49]:
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

from src.processing.vector_store import (
    get_or_create_collection,
)

from pathlib import Path


@st.cache_resource
def load_agent_service():

    project_root = Path.cwd()

    vector_store_path = (
        project_root
        / "data"
        / "vector_store"
    )

    collection = get_or_create_collection(
        collection_name="alessia_collection",
        vector_store_path=vector_store_path,
    )

    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    reranker_model = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L-6-v2"
    )

    return AgentService(
        collection=collection,
        embedding_model=embedding_model,
        reranker_model=reranker_model,
    )

### 6. Inicialización del servicio

Se crea la instancia de AgentService que será utilizada por la interfaz para procesar las consultas del usuario.

In [50]:
agent_service = load_agent_service()